# Śledzenie obiektów

<img src="https://i.imgur.com/wKXXFkQ.png" width="500">

## Wstęp
W erze cyfrowej, w obliczu rosnącej lawinowo ilości danych wideo, zdolność do ich automatycznego rozpoznawania i interpretowania staje się kluczowa w wielu dziedzinach – od bezpieczeństwa publicznego po autonomiczne pojazdy. Technologie oparte na głębokim uczeniu rewolucjonizują sposób, w jaki przetwarzamy informacje wizualne. Kluczowym wyzwaniem jest tu detekcja i śledzenie obiektów na filmach wideo.

Celem tego zadania jest opracowanie algorytmu, który będzie w stanie analizować sekwencje ruchów w grze "trzy kubki". Uczestnicy mają za zadanie określić końcową pozycję kubków po serii ruchów, korzystając z analizy statycznych obrazów z każdej klatki nagrania.

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI PODCZAS WYSYŁANIA ##########################

# Poniższe funkcje ułatwiają pracę z dostarczonymi danymi
# W kolejnych komórkach zobaczysz przykłady ich użycia
from utils.utils import get_level_info, get_video_data, display_video, download_and_replace_data

FINAL_EVALUATION_MODE = False
# W czasie sprawdzania Twojego rozwiązania, zmienimy tę wartość na True
# Wartość tej flagi M U S I zostać ustawiona na False w rozwiązaniu, które nam nadeślesz!

images, coordinates, target, path_to_images = get_video_data(level=2,video_id=0,dataset="example")
display_video(images,rescale=0.7,FINAL_EVALUATION_MODE=FINAL_EVALUATION_MODE)

## Zadanie 2: Okluzje, rozmycia, przesłonienie

Celem zadania jest opracowanie algorytmu, który potrafi przetwarzać sekwencje obrazów z gry w trzy kubki, nawet gdy występują rozmycia czy przesłonięcia. Zadanie ma na celu nauczenie maszyny wykorzystywania ciągłości informacji z kolejnych klatek, aby mimo chwilowych utrudnień w percepcji, mogła skutecznie określić końcową pozycję kubków.

Napisz algorytm który poradzi sobie z trudnieszym zbiorem danych - `Level_2`. Składa się on z animacji, w których pojawiają się dodatkowe utrudnienia:
- Przesłonięcia obiektu przez inny, powodujące, że są one traktowane jako jeden,
- Prostokąty ograniczające nie są już idealnie dopasowane do obiektów,
- Prostokąty ograniczające nie są widoczne we wszystkich klatkach.

Będziesz miał dostęp zarówno do wszystkich klatek animacji, jak i do oznaczonych przez nas prostokątów ograniczających, w których znajdują się kubki. Co ważne, algorytm, który będziesz tworzył ma korzystać jedynie z informacji o prostokątach ograniczających. W tym zadaniu, klatki wideo są dostarczone jedynie do wizualizacji przykładów i algorytmu, na własne potrzeby.

Punkty za to zadanie będą przyznane za osiągnięcie jak najdokładniejszych predykcji na zbiorze testowym. Kryterium będzie *accuracy* i spodziewamy się wyników powyżej `80%`. Ewaluacja na zbiorze testowym będzie dokonana przez organizatorów.

## Pliki zgłoszeniowe
Tylko ten notebook zawierający **kod** oraz **krótki raport** opisujący Twoje rozwiązanie (do 300 słów). Miejsce na raport znajdziesz na końcu tego notebooka.

## Ograniczenia
- Twoja funkcja powinna zwracać predykcje w maksymalnie 5 minut używając Google Colab bez GPU.

## Uwagi i wskazówki
- Testuj swoje rozwiązanie na zbiorze plików wideo `level_2`.
- **Skuteczność modelu**: przetestuj skuteczność modelu na zbiorze walidacyjnym używając dostarczonej przez nas funkcji **submission_script**, umieść ten wynik w raporcie.

## Ewaluacja
Pamiętaj, że podczas sprawdzania flaga `FINAL_EVALUATION_MODE` zostanie ustawiona na `True`. Za pomocą skryptu `validation_script.py` możesz upewnić się, że Twoje rozwiązanie zostanie prawidłowo wykonane na naszych serwerach oceniających.

Za to podzadanie możesz zdobyć pomiędzy 0 i 0.5 punktów. Zdobędziesz 0 punktów jeśli Twoje accuracy na zbiorze testowym będzie poniżej 50%. Jeśli będzie większe niż 95%, otrzymasz 0.5 punktu. Pomiędzy tymi wartościami, wynik rośnie liniowo z wartością metryki.

# Kod startowy

In [ ]:
# Poniższe biblioteki są wystarczające do wykonania wszystkich zadań
# Jeśli jednak chcesz użyć innych, sprawdź czy są dostępne na serwerze (requirements.txt)
import numpy as np
import os
import matplotlib.pyplot as plt
import torch
import IPython.display
import json
import PIL
import sklearn as sk

In [ ]:
# funkcja pomocnicza do ładowania danych
images, _, _, _ = get_video_data(level=2,video_id=0,dataset="example")

with open(os.path.join(os.getcwd(),'example_tracks','tracks_2_0.json'), 'r') as f:
    tracks = json.load(f)

for key in tracks.keys():
    tracks[key] = [tuple(el) for el in tracks[key]]

# funkcja pomocnicza do wyświetlania danych
display_video(images,
                tracks=tracks,
                rescale=0.7,
                FINAL_EVALUATION_MODE=FINAL_EVALUATION_MODE)

In [ ]:
# Pobieranie danych do podzadań 1, 2 i 3 (około ~646Mb), skrypt będzie wykonywał się parę minut
# Wystarczy, że pobierzesz dane tylko raz. Na serwerze sprawdzającym dane będą już pobrane,
# struktura plików będzie identyczna jak tutaj
if not FINAL_EVALUATION_MODE:
    download_and_replace_data()

In [ ]:
######################### NIE ZMIENIAJ TEJ KOMÓRKI ##########################

# funkcja pomocnicza do testowania algorytmu
def submission_script(algorithm,level,verbose=False,dataset="valid"):
    num_videos, _ = get_level_info(level=level,dataset=dataset)
    correct = []
    exception_messages = set()
    for video_number in range(num_videos):
        _, coordinates, target, _ = get_video_data(level=level,video_id=video_number,dataset=dataset)
        try:
            prediction = algorithm(coordinates)
            if tuple(target) == tuple(prediction):
                correct.append(1)
            else:
                correct.append(0)
            if verbose:
                print(f"Video: animation_{str(video_number).zfill(4)}")
                print(f"Prediction: {prediction}")
                print(f"Target:     {target}")
                print(f"Score: {tuple(target) == tuple(prediction)}", end='\n\n')
        except Exception as e:
            correct.append(0)
            exception_messages.add(str(e))
    if verbose:
        print(f"Accuracy: {np.mean(correct)}")
        print(f"Correctness: {correct}")
    return np.sum(correct) / num_videos, correct, exception_messages

# Twoje rozwiązanie

In [ ]:
import math

# v_t = 0.85, a_t = 2.0, sqrt = True, vel_points_1 = [[6, 1.0], [3, 100.0]], vel_points_2 = [[6, 1.0], [5, 5.0]], accel_points = [[6, 1.0], [2, 5.0]]

vel_points_1 = [[6, 1.0], [3, 3.0]]
vel_points_2 = [[6, 1.0], [5, 6.0]]
accel_points = [[6, 1.0], [2, 5.0]]
v_t = 0.9
a_t = 2.0

permutations = [[0, 1, 2], [0, 2, 1], [1, 0, 2], [1, 2, 0], [2, 0, 1], [2, 1, 0]]

def get_error(avg1, avg2):
    if len(avg1) == 0:
        return 0
    else:
        return np.sqrt(((avg1 - avg2) ** 2).sum(axis=1)).sum()

def your_algorithm_task_2(coordinates): # nie zmieniaj nazwy funkcji
    avgs = []
    for frame_name in coordinates:
        cord = np.array(coordinates[frame_name])
        avg = [[(crd[2] + crd[0]) / 2.0, (crd[3] + crd[1]) / 2.0] for crd in cord]
        avg.sort()
        avgs.append(np.array(avg))

    crds = np.copy(avgs[0])
    positions = []
    velocities_1 = []
    velocities_2 = []

    def calc_vel(k, t, vel_points):
        totalWeight = 0
        sum = [0, 0]
        for dt, weight in vel_points:
            if t - dt >= 0:
                pos1 = positions[t - dt][k]
                pos2 = positions[t][k]
                totalWeight += weight
                if dt > 0:
                    sum[0] += ((pos2[0] - pos1[0]) / dt) * weight
                    sum[1] += ((pos2[1] - pos1[1]) / dt) * weight
        if totalWeight > 0:
            return [sum[0] / totalWeight, sum[1] / totalWeight]
        else:
            return [0, 0]

    def calc_accel(k, t):
        totalWeight = 0
        sum = [0, 0]
        for dt, weight in accel_points:
            if t - dt >= 0:
                vel1 = velocities_2[t - dt][k]
                vel2 = velocities_2[t][k]
                totalWeight += weight
                if dt > 0:
                    sum[0] += ((vel2[0] - vel1[0]) / dt) * weight
                    sum[1] += ((vel2[1] - vel1[1]) / dt) * weight
        if totalWeight > 0:
            return [sum[0] / totalWeight, sum[1] / totalWeight]
        else:
            return [0, 0]

    for t, frm in enumerate(avgs):
        positions.append(np.copy(crds))

        vel = []
        for k in range(3):   
            vel.append(calc_vel(k, t, vel_points_1))
        velocities_1.append(np.array(vel))

        vel = []
        for k in range(3):   
            vel.append(calc_vel(k, t, vel_points_2))
        velocities_2.append(np.array(vel))

        accel = []
        for k in range(3):   
            accel.append(calc_accel(k, t))
        accel = np.array(accel)

        expected_velocities = velocities_1[t] + a_t * accel
        expected_crds = crds + v_t * expected_velocities

        bestError = 100000
        for perm in permutations:
            err = get_error(frm, [crds[perm[i]] for i in range(len(frm))])
            if err < bestError:
                bestError = err;
                bestPerm = perm;
        for k in range(0, len(frm)):
            crds[bestPerm[k]] = frm[k]
        for k in range(len(frm), 3):
            crds[bestPerm[k]] = expected_crds[bestPerm[k]]

    xs = crds[:, 0]
    idx1 = xs.argmin()
    xs[idx1] = 1000000
    idx2 = xs.argmin()
    xs[idx2] = 1000000
    idx3 = xs.argmin()

    return [idx1, idx2, idx3]

In [ ]:
# zapisz swój raport do zmiennej poniżej, abyśmy mogli go później automatycznie odczytać sprawdzaczką
raport_2 = \
"""
Raport z zadania:
Podobnie jak w pierwszym podzadaniu, algorytm będzie pracować tylko na środkach prostokątów i będzie analizować, które przyporządkowanie starych pozycji do nowych jest najlepsze.
Aby policzyć jakość przyporządkowania, zdecydowałem się policzyć sumę odległości euklidesowych między starymi pozycjami a nowymi zamiast MSE.
Poza tym, jedyną różnicą względem pierwszego podzadania jest to, co algorytm robi w przypadku, gdy w animacji znajdują się mniej niż 3 prostokąty ograniczające:
 - Dla każdego kubka, w każdej klatce algorytm liczy dwa zestawy prędkości (jako wektorów) tych kubków. Prędkości te liczy jako v = (p(t) - p(t - dt)) / dt, gdzie p(t) to pozycja kubka w chwili 't'.
 - Pierwszy zestaw jest średnią ważoną prędkości policzonych dla dt=6 i dt=3 z wagami kolejno 1.0 i 3.0 (wagi oraz wartości 'dt' zostały dopasowane przez algorytm szukający najwyższej accuracy na zbiorze treningowym).
 - Drugi zestaw jest policzony dla dt=6 i dt=3 z wagami 1.0 i 6.0
 - Następnie, liczymy przyspieszenie jako a = (v(t) - v(t - dt)) / dt na podstawie prędkości z drugiego zestawu. Przyspieszenie jest policzone dla dt=6 i dt=2 z wagami 1.0 i 5.0
 - Gdy pozycja jednego z kubków nie jest znana (mniej niż 3 prostokąty ograniczające), to użyjemy pozycji policzonej jako p(t + 1) = p(t) + v_t * (v(t) + a_t * a(t)), gdzie v_t = 0.85 i a_t = 2.0 są parametrami znalezionymi w ten sam sposób co wagi.
Tyle wystarczy, aby algorytm uzyskał 88.7% dokładności na wszystkich danych dostępnych w zadaniu (88% na zbiorze treningowym i 90% na testowym)
"""